In [ ]:
%py
# PySpark script to mask the last 4 digits of invoice_number in the d_product_revenue_clone table

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring, lit, concat

# Create a Spark session
spark = SparkSession.builder \
    .appName("Masking Invoice Number") \
    .getOrCreate()

# Drop table if it exists
spark.sql("DROP TABLE IF EXISTS purgo_playground.d_product_revenue_clone")

# Create a clone of the d_product_revenue table
spark.sql("""
    CREATE TABLE purgo_playground.d_product_revenue_clone AS
    SELECT * FROM purgo_playground.d_product_revenue
""")

# Load the cloned table into a DataFrame
df_clone = spark.table("purgo_playground.d_product_revenue_clone")

# Mask the last 4 digits of the invoice_number column
df_masked = df_clone.withColumn(
    "invoice_number",
    concat(substring(col("invoice_number").cast("string"), 1, 6), lit("****"))
)

# Save the DataFrame back to the cloned table
df_masked.write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")

# Validate that the invoice numbers are masked correctly
df_result = spark.table("purgo_playground.d_product_revenue_clone").select("invoice_number")
df_result.show()

# Stop the Spark session
spark.stop()